In [ ]:
# ============================================================
# ТРАНСПОРТНАЯ ЗАДАЧА ЛИНЕЙНОГО ПРОГРАММИРОВАНИЯ
# ============================================================
# Имеется m=3 поставщика P_i и n=3 потребителя D_j.
# Переменные решения: x_ij >= 0 — объём поставки от P_i к D_j.
#
# Целевая функция (минимизация суммарных затрат):
#   min Z = sum_{i=1}^{m} sum_{j=1}^{n} c_ij * x_ij
#
# Ограничения по предложению (лимит поставщика P_i):
#   sum_{j=1}^{n} x_ij <= s_i,  i = 1..m
#
# Ограничения по спросу (спрос устройства D_j удовлетворяется полностью):
#   sum_{i=1}^{m} x_ij = d_j,   j = 1..n
#
# Условие допустимости (необходимое): sum(s_i) >= sum(d_j)
#
# Матричная форма стандартной задачи ЛП:
#   min c^T x,  A_ub x <= b_ub,  A_eq x = b_eq,  x >= 0
# где x = vec(X) — построчная развёртка матрицы поставок X в вектор длины m*n.
import numpy as np
import time
from scipy.optimize import linprog

In [ ]:
# Матрица затрат C in R^{m x n}: c_ij = стоимость (руб.) поставки 1 ед. от P_i к D_j.
# Входит в целевую функцию как: Z = sum_i sum_j c_ij * x_ij = c^T * vec(X)
# (скалярное произведение развёрток матриц C и X).
# Строки — поставщики P1, P2, P3; столбцы — устройства D1, D2, D3.
C = np.array([
    [120, 130, 140],  # P1: c_11=120, c_12=130, c_13=140
    [135, 125, 138],  # P2: c_21=135, c_22=125, c_23=138
    [150, 145, 132],  # P3: c_31=150, c_32=145, c_33=132
], dtype=np.float64)

print("Матрица затрат C (руб./ед.):")
print(f"{'':>5} {'D1':>6} {'D2':>6} {'D3':>6}")
for i, row in enumerate(C, 1):
    print(f"P{i}:  {row[0]:>6.0f} {row[1]:>6.0f} {row[2]:>6.0f}")

In [ ]:
# Вектор предложения s in R^m: s_i — максимальный объём поставки поставщика P_i (ед./мес.).
# Формирует правую часть ограничений-неравенств:
#   sum_{j=1}^{n} x_ij <= s_i  для каждого i = 1,2,3
# В scipy.optimize.linprog передаётся как аргумент b_ub.
b_ub = np.array([5000, 4000, 3500], dtype=np.float64)
print("b_ub (лимиты поставщиков):", b_ub)

In [ ]:
# Вектор спроса d in R^n: d_j — обязательный объём поставок устройству D_j (ед./мес.).
# Формирует правую часть ограничений-равенств:
#   sum_{i=1}^{m} x_ij = d_j  для каждого j = 1,2,3
# В scipy.optimize.linprog передаётся как аргумент b_eq.
#
# Условие допустимости: sum(s_i) >= sum(d_j).
# При выполнении строгого равенства задача называется сбалансированной
# (closed transportation problem); излишек предложения не расходуется.
b_eq = np.array([3000, 4200, 2500], dtype=np.float64)
print("b_eq (спрос на устройства):", b_eq)
print(f"Суммарный спрос: {b_eq.sum():.0f}  |  Суммарное предложение: {b_ub.sum():.0f}  → задача допустима")

In [ ]:
# Построчная развёртка матрицы C в вектор c in R^{m*n} (операция vec по строкам):
#   c = [c_11, c_12, c_13, c_21, c_22, c_23, c_31, c_32, c_33]
#
# Это приводит целевую функцию к стандартной форме ЛП:
#   Z = c^T * x,  где x = [x_11, x_12, x_13, x_21, ..., x_33]^T in R^9
#
# Соответствие индексов: позиция k = (i-1)*n + (j-1) в векторе x
# соответствует переменной x_ij (нумерация с 0).
c = C.flatten()
labels = [f"x{i}{j}" for i in range(1,4) for j in range(1,4)]
print("Вектор c:", dict(zip(labels, c.astype(int))))

In [ ]:
# Матрица ограничений-неравенств A_ub in {0,1}^{m x m*n}.
#
# Строка i соответствует поставщику P_i.
# Элемент (A_ub)_{i,k} = 1, если k = (i-1)*n + j для некоторого j in {0..n-1}, иначе 0.
# Таким образом, i-я строка выбирает все переменные x_ij одного поставщика.
#
# Ограничение i-й строки: (A_ub * x)[i] <= b_ub[i], то есть:
#   x_i1 + x_i2 + x_i3 <= s_i
#
# Структура A_ub — блочно-диагональная: i-й блок 1×n стоит в столбцах [(i-1)*n .. i*n).
#
#  Переменные: x11 x12 x13 | x21 x22 x23 | x31 x32 x33
A_ub = np.array([
    [1,  1,  1,  0,  0,  0,  0,  0,  0],  # P1: x11+x12+x13 <= 5000
    [0,  0,  0,  1,  1,  1,  0,  0,  0],  # P2: x21+x22+x23 <= 4000
    [0,  0,  0,  0,  0,  0,  1,  1,  1],  # P3: x31+x32+x33 <= 3500
], dtype=np.float64)

print("Матрица A_ub (3×9):")
print(f"{'':>4}", '  '.join(f"{l:>4}" for l in labels), "  b_ub")
for i, (row, b) in enumerate(zip(A_ub, b_ub), 1):
    print(f"P{i}: ", '  '.join(f"{int(v):>4}" for v in row), f"  {b:.0f}")

In [ ]:
# Матрица ограничений-равенств A_eq in {0,1}^{n x m*n}.
#
# Строка j соответствует устройству D_j.
# Элемент (A_eq)_{j,k} = 1, если k = (i-1)*n + (j-1) для некоторого i in {0..m-1}, иначе 0.
# Таким образом, j-я строка выбирает все переменные x_ij одного потребителя.
#
# Ограничение j-й строки: (A_eq * x)[j] = b_eq[j], то есть:
#   x_1j + x_2j + x_3j = d_j
#
# Единицы расположены через шаг n: позиции (j-1), (j-1)+n, (j-1)+2n (нумерация с 0).
#   j=1 → позиции 0, 3, 6
#   j=2 → позиции 1, 4, 7
#   j=3 → позиции 2, 5, 8
#
#  Переменные: x11 x12 x13 | x21 x22 x23 | x31 x32 x33
A_eq = np.array([
    [1,  0,  0,  1,  0,  0,  1,  0,  0],  # D1: x11+x21+x31 = 3000
    [0,  1,  0,  0,  1,  0,  0,  1,  0],  # D2: x12+x22+x32 = 4200
    [0,  0,  1,  0,  0,  1,  0,  0,  1],  # D3: x13+x23+x33 = 2500
], dtype=np.float64)

print("Матрица A_eq (3×9):")
print(f"{'':>4}", '  '.join(f"{l:>4}" for l in labels), "  b_eq")
for j, (row, b) in enumerate(zip(A_eq, b_eq), 1):
    print(f"D{j}: ", '  '.join(f"{int(v):>4}" for v in row), f"  {b:.0f}")

In [ ]:
# Решение задачи ЛП методами библиотеки HiGHS.
#
# 'highs'     — автоматический выбор алгоритма (препроцессинг + симплекс или IPM).
#
# 'highs-ds'  — двойственный симплекс (Dual Simplex Method):
#               на каждой итерации выбирается входящая переменная по правилу
#               минимального коэффициента двойственного нарушения;
#               сложность в худшем случае O(2^n), на практике — полиномиальна.
#
# 'highs-ipm' — метод внутренней точки (Interior Point / барьерный метод):
#               минимизирует логарифмический барьер:
#                   F(x, mu) = c^T x - mu * sum_k ln(x_k),  mu → 0
#               гарантированная полиномиальная сложность O(n^3.5 * L),
#               где L — длина битового представления входных данных.
#
# bounds=(0, None): x_ij >= 0, верхних ограничений на отдельные поставки нет.
methods = ['highs', 'highs-ds', 'highs-ipm']
results = {}
times = {}

for method in methods:
    t0 = time.perf_counter()
    res = linprog(
        c=c,
        A_ub=A_ub, b_ub=b_ub,
        A_eq=A_eq, b_eq=b_eq,
        bounds=(0, None),
        method=method
    )
    dt = time.perf_counter() - t0
    results[method] = res
    times[method] = dt
    print(f"Метод '{method}': статус={'Успех' if res.success else 'Ошибка'}  "
          f"Z = {res.fun:.2f} руб.  время = {dt*1000:.4f} мс")

In [ ]:
# Сравнительная таблица методов.
# Все три метода дают одно значение Z* — это следствие теоремы:
# задача ЛП на непустом ограниченном многограннике имеет единственный оптимум
# (если многогранник невырожден). Методы различаются только вычислительным путём к нему.
print(f"{'Метод':<12} {'Статус':<10} {'Z, руб.':<18} {'Время, мс'}")
print("-" * 50)
for m in methods:
    r = results[m]
    print(f"{m:<12} {'Успех' if r.success else 'Ошибка':<10} {r.fun:<18.2f} {times[m]*1000:.4f}")

fastest = min(times, key=times.get)
print(f"\nСамый быстрый метод: '{fastest}' ({times[fastest]*1000:.4f} мс)")

In [ ]:
# Интерпретация оптимального решения x* in R^9.
#
# reshape(m, n) — обратная операция к vec():
#   восстанавливает матрицу поставок X* in R^{m x n},  X*[i,j] = x*_{i+1, j+1}.
#
# Проверка допустимости оптимума:
#   строчные суммы:   sum_j X*[i,j] <= s_i  (лимит поставщика не превышен)
#   столбцовые суммы: sum_i X*[i,j] = d_j   (спрос удовлетворён полностью)
#
# Вклад ненулевых поставок в целевую функцию:
#   dZ_ij = c_ij * x*_ij
#   Z* = sum_{i,j: x*_ij > 0} dZ_ij
res = results['highs']
x = res.x.reshape(3, 3)
suppliers = ['P1', 'P2', 'P3']
devices   = ['D1', 'D2', 'D3']

print("Оптимальный план закупок X* (ед./мес.):")
print(f"{'':>5} {'D1':>8} {'D2':>8} {'D3':>8} {'Итого':>8}")
for i, sup in enumerate(suppliers):
    print(f"{sup}:  {x[i,0]:>8.1f} {x[i,1]:>8.1f} {x[i,2]:>8.1f} {x[i].sum():>8.1f}  (лимит: {b_ub[i]:.0f})")
print(f"{'Спрос':>5}: {'3000':>8} {'4200':>8} {'2500':>8}")

print("\nНенулевые поставки:")
for i, sup in enumerate(suppliers):
    for j, dev in enumerate(devices):
        if x[i, j] > 0.01:
            print(f"  {sup} → {dev}: {x[i,j]:.0f} ед. × {C[i,j]:.0f} руб. = {x[i,j]*C[i,j]:.0f} руб.")

print(f"\nМинимальные суммарные затраты: {res.fun:.2f} руб./мес.")